In [1]:
from pylib.setup import *
setup_notebook()

from pylib.assumptions import smr_tech, SMR_DOWNTIME, solar_tech, wind_tech, battery, GRID_TARIFF_BUY, GRID_TARIFF_SELL, grid_connect_annual
from pylib.model import DatacenterDemand, GridSupply, KKSupply, VESupply

# Scenario analysis: KK vs VE — 2022–2025, x ∈ {25 %, 50 %, 75 %}

*`1. configuration`*

In [2]:
# 1. konfiguration
YEARS         = [2025]
X_VALUES      = [0.30, 0.50, 0.70, 0.90]
STORAGE_HOURS = 4

# Installed capacity per year — from 1_input.ipynb VE() output
# "Index denominator is: SolarPowerCapacity / OnshoreWindCapacity: <value> (last observed)"
SOLAR_CAP_MW = {2022: 2_752.88, 2023: 3_528.93, 2024: 3_913.24, 2025: 4_955.54}
WIND_CAP_MW  = {2022: 4_715.28, 2023: 4_801.21, 2024: 4_861.83, 2025: 4_878.48}

*`2. model runs`*

In [3]:
# 2. model runs  (12 LP solves — ~seconds total)
pat  = pathlib.Path('variation_patterns')
rows = []

for year in YEARS:
    solar_cf = np.loadtxt(pat / f'PV_VE_{year}_{year+1}.txt') / SOLAR_CAP_MW[year]
    wind_cf  = np.loadtxt(pat / f'WL_VE_{year}_{year+1}.txt') / WIND_CAP_MW[year]
    prices   = np.loadtxt(pat / f'wp_{year}_{year+1}.txt')    / 7.46

    print(f'{year}  priser: mean={prices.mean():.1f}  min={prices.min():.1f}  max={prices.max():.1f} EUR/MWh')

    for x in X_VALUES:
        demand = DatacenterDemand(demand_mw=1_000.0, x=x)
        grid   = GridSupply(prices, demand)
        kk     = KKSupply(smr_tech, demand, prices, downtime_fraction=SMR_DOWNTIME,
                          buy_tariff=GRID_TARIFF_BUY,
                          grid_connect_annual=grid_connect_annual(demand.demand_mw))
        ve     = VESupply(solar_cf, wind_cf, solar_tech, wind_tech, battery,
                          demand, prices=prices, storage_hours=STORAGE_HOURS,
                          buy_tariff=GRID_TARIFF_BUY, sell_tariff=GRID_TARIFF_SELL,
                          grid_connect_annual=grid_connect_annual(demand.grid_cap_mw))

        r_kk = kk.result(grid)
        r_ve = ve.result(grid)

        rows.append({
            'år':             year,
            'x':              x,
            'kk_lcoe':        r_kk.lcoe,
            've_lcoe':        r_ve.lcoe,
            'gap_lcoe':       r_ve.lcoe - r_kk.lcoe,
            'kk_total_meur':  r_kk.annual_total_cost / 1e6,
            've_total_meur':  r_ve.annual_total_cost / 1e6,
            'gap_meur':      (r_ve.annual_total_cost - r_kk.annual_total_cost) / 1e6,
            've_export_meur': r_ve.annual_export_revenue / 1e6,
            've_grid_meur':   r_ve.annual_grid_cost / 1e6,
        })
        print(f'  x={x:.0%}  KK {r_kk.lcoe:.2f}  VE {r_ve.lcoe:.2f}  gap {r_ve.lcoe - r_kk.lcoe:+.2f} €/MWh')

df = pd.DataFrame(rows)
print('\nFærdig.')

2025  priser: mean=81.6  min=-30.7  max=583.5 EUR/MWh
  x=30%  KK 75.39  VE 71.38  gap -4.01 €/MWh
  x=50%  KK 75.39  VE 101.41  gap +26.02 €/MWh
  x=70%  KK 75.39  VE 143.93  gap +68.54 €/MWh
  x=90%  KK 75.39  VE 199.26  gap +123.87 €/MWh

Færdig.


*`3. results table`*

In [4]:
# 3. resultattabel
tbl = (
    df
    .assign(x=lambda d: d['x'].map(lambda v: f'{v:.0%}'))
    .set_index(['år', 'x'])
    [['kk_total_meur', 've_total_meur', 'gap_meur']]
    .rename(columns={
        'kk_total_meur': 'KK (M€/yr)',
        've_total_meur': 'VE (M€/yr)',
        'gap_meur':      'VE−KK (M€/yr)',
    })
)

tbl.style.format({
    'KK (M€/yr)':     '{:.1f}',
    'VE (M€/yr)':     '{:.1f}',
    'VE−KK (M€/yr)':  '{:+.1f}',
})